# Assignment — Holistic Prompt Optimization

**Program:** Brilian Sistem Informasi Bootcamp  
**Session 25:** Prompt Optimization

## 🎯 Learning Objectives

By completing this assignment, you will:
1. Apply **Chain-of-Thought**, **Self-Check**, and **Retrieval-Aware** prompting on real automotive scenarios
2. Combine reasoning techniques with **engine tuning** (Temperature, Top-P, Top-K) and **cost optimization** (max_tokens, prompt compression)
3. Analyze trade-offs between Safe / Balanced / Creative parameter configurations

## 📅 Submission

| Item | Detail |
|---|---|
| Platform | Google Classroom |
| File naming | `NamaLengkap_Sesi25_Assignment.ipynb` |
| Deadline | _[insert deadline]_ |
| Late policy | _[insert policy]_ |

## ⚙️ Setup

> ⚠️ **IMPORTANT — Never hardcode your API key.** Always use Google Colab Secrets.

**How to set it up:**
1. Click the 🔑 icon on the left sidebar in Colab
2. Add a secret named `GOOGLE_API_KEY` with your Gemini API key from [Google AI Studio](https://aistudio.google.com/)
3. Toggle **Notebook access** ON


In [15]:
!pip install -q google-generativeai


In [16]:
import google.generativeai as genai
from google.colab import userdata
import textwrap

# 🔐 Always retrieve secrets from Colab Secrets — never hardcode
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    print("✅ Gemini API configured successfully.")
except Exception as e:
    print(f"❌ Error: {e}")
    print("Please ensure GOOGLE_API_KEY is set in Colab Secrets.")


✅ Gemini API configured successfully.


## 🛠️ Helper Function

This helper supports all engine parameters PLUS reports token usage so you can analyze cost efficiency.


In [17]:
def generate(prompt, temp=0.7, top_p=0.9, top_k=40, max_tokens=512, label=""):
    """Generate text and report token usage for cost analysis."""
    model = genai.GenerativeModel('gemini-3.1-flash-lite')

    config = genai.types.GenerationConfig(
        temperature=temp,
        top_p=top_p,
        top_k=top_k,
        max_output_tokens=max_tokens,
    )

    response = model.generate_content(prompt, generation_config=config)

    print(f"\n{'=' * 70}")
    print(f"  {label}")
    print(f"  Params: temp={temp}, top_p={top_p}, top_k={top_k}, max_tokens={max_tokens}")
    print(f"{'=' * 70}")
    print(textwrap.fill(response.text, width=80))

    if hasattr(response, 'usage_metadata') and response.usage_metadata:
        m = response.usage_metadata
        print(
            f"\n📊 Tokens — input: {m.prompt_token_count}, "
            f"output: {m.candidates_token_count}, total: {m.total_token_count}"
        )
    return response.text


---

# Section 1 — Chain-of-Thought (CoT)

**Skenario:** Tentukan harga trade-in untuk **Mitsubishi Xpander Ultimate 2019 A/T**, KM 65.000, pajak hidup sampai Agustus 2026, ada lecet di bumper depan, service history lengkap di bengkel resmi. Harga pasaran sejenis Rp 195–215jt.

Bandingkan output **tanpa CoT** vs **dengan CoT**.


In [18]:
# --- WITHOUT CoT ---
basic_prompt = """
Tentukan harga trade-in untuk Mitsubishi Xpander Ultimate 2019 A/T,
KM 65.000, pajak hidup sampai Agustus 2026, ada lecet di bumper depan,
service history lengkap di bengkel resmi. Harga pasaran sejenis Rp 195-215jt.
Berapa rekomendasi harga trade-in?
"""

generate(basic_prompt, temp=0.3, label="Section 1 — WITHOUT CoT")



  Section 1 — WITHOUT CoT
  Params: temp=0.3, top_p=0.9, top_k=40, max_tokens=512
Untuk menentukan harga *trade-in* (tukar tambah), Anda harus memahami bahwa
dealer akan membeli mobil Anda di bawah harga pasaran (harga jual ke konsumen)
karena mereka harus memperhitungkan biaya perbaikan (*reconditioning*), margin
keuntungan, dan risiko stok.  Berikut adalah analisis estimasi harga untuk
Mitsubishi Xpander Ultimate 2019 Anda:  ### 1. Analisis Kondisi Mobil *
**Positif:**     *   **Pajak Panjang:** Pajak hidup sampai Agustus 2026 adalah
nilai tambah yang sangat besar (menghemat biaya pembeli berikutnya sekitar Rp
4-5 juta).     *   **Service Record:** Riwayat servis di bengkel resmi adalah
poin krusial yang menjaga harga jual tetap stabil. *   **Negatif:**     *   **KM
65.000:** Termasuk kategori pemakaian wajar/sedikit tinggi untuk usia 5 tahun
(rata-rata 13rb km/tahun).     *   **Lecet Bumper:** Ini akan menjadi poin
negosiasi dealer untuk memotong harga (biaya cat ulang bumper biasa

'Untuk menentukan harga *trade-in* (tukar tambah), Anda harus memahami bahwa dealer akan membeli mobil Anda di bawah harga pasaran (harga jual ke konsumen) karena mereka harus memperhitungkan biaya perbaikan (*reconditioning*), margin keuntungan, dan risiko stok.\n\nBerikut adalah analisis estimasi harga untuk Mitsubishi Xpander Ultimate 2019 Anda:\n\n### 1. Analisis Kondisi Mobil\n*   **Positif:**\n    *   **Pajak Panjang:** Pajak hidup sampai Agustus 2026 adalah nilai tambah yang sangat besar (menghemat biaya pembeli berikutnya sekitar Rp 4-5 juta).\n    *   **Service Record:** Riwayat servis di bengkel resmi adalah poin krusial yang menjaga harga jual tetap stabil.\n*   **Negatif:**\n    *   **KM 65.000:** Termasuk kategori pemakaian wajar/sedikit tinggi untuk usia 5 tahun (rata-rata 13rb km/tahun).\n    *   **Lecet Bumper:** Ini akan menjadi poin negosiasi dealer untuk memotong harga (biaya cat ulang bumper biasanya Rp 500rb - 1jt per panel).\n\n### 2. Estimasi Harga Trade-in\nJika

In [19]:
# --- WITH CoT ---
cot_prompt = """
Tentukan harga trade-in untuk Mitsubishi Xpander Ultimate 2019 A/T,
KM 65.000, pajak hidup sampai Agustus 2026, ada lecet di bumper depan,
service history lengkap di bengkel resmi. Harga pasaran sejenis Rp 195-215jt.

Lakukan analisis bertahap berikut sebelum memberikan rekomendasi:
1. Identifikasi faktor-faktor yang memengaruhi harga (tahun, KM, kondisi fisik, service history, status pajak)
2. Bandingkan dengan rentang harga pasaran
3. Hitung depresiasi & adjustment kondisi (potongan untuk lecet bumper, premium untuk service history lengkap)
4. Berikan rekomendasi harga trade-in dengan justifikasi setiap angka

Format jawaban: Langkah 1, Langkah 2, Langkah 3, Langkah 4, lalu Rekomendasi Akhir.
"""

generate(cot_prompt, temp=0.3, label="Section 1 — WITH CoT")



  Section 1 — WITH CoT
  Params: temp=0.3, top_p=0.9, top_k=40, max_tokens=512
Berikut adalah analisis estimasi harga *trade-in* untuk Mitsubishi Xpander
Ultimate 2019 A/T Anda:  ### Langkah 1: Identifikasi Faktor yang Memengaruhi
Harga *   **Tahun (2019):** Masuk kategori mobil usia 5 tahun, di mana
depresiasi mulai melambat namun tetap signifikan. *   **KM (65.000):** Angka ini
tergolong *wajar/normal* untuk pemakaian 5 tahun (rata-rata 13.000 km/tahun).
Tidak terlalu rendah, tidak terlalu tinggi. *   **Kondisi Fisik (Lecet
Bumper):** Merupakan *minor defect* yang akan menjadi poin negosiasi bagi dealer
untuk biaya pengecatan ulang (re-paint). *   **Service History:** Sangat
positif. Riwayat di bengkel resmi meningkatkan nilai jual (trust) dan memudahkan
proses pengecekan kondisi mesin bagi pembeli berikutnya. *   **Pajak (Agustus
2026):** Sangat positif. Pajak yang baru saja dibayar (panjang) adalah nilai
tambah yang signifikan karena pembeli tidak perlu mengeluarkan biaya tambahan

'Berikut adalah analisis estimasi harga *trade-in* untuk Mitsubishi Xpander Ultimate 2019 A/T Anda:\n\n### Langkah 1: Identifikasi Faktor yang Memengaruhi Harga\n*   **Tahun (2019):** Masuk kategori mobil usia 5 tahun, di mana depresiasi mulai melambat namun tetap signifikan.\n*   **KM (65.000):** Angka ini tergolong *wajar/normal* untuk pemakaian 5 tahun (rata-rata 13.000 km/tahun). Tidak terlalu rendah, tidak terlalu tinggi.\n*   **Kondisi Fisik (Lecet Bumper):** Merupakan *minor defect* yang akan menjadi poin negosiasi bagi dealer untuk biaya pengecatan ulang (re-paint).\n*   **Service History:** Sangat positif. Riwayat di bengkel resmi meningkatkan nilai jual (trust) dan memudahkan proses pengecekan kondisi mesin bagi pembeli berikutnya.\n*   **Pajak (Agustus 2026):** Sangat positif. Pajak yang baru saja dibayar (panjang) adalah nilai tambah yang signifikan karena pembeli tidak perlu mengeluarkan biaya tambahan dalam waktu dekat.\n\n### Langkah 2: Perbandingan dengan Rentang Harga 

### 🔍 Reflection — Section 1

_(Tulis jawaban singkat di sel markdown ini setelah menjalankan kedua versi)_

1. Faktor apa yang muncul di output CoT yang **tidak** muncul di output basic?
2. Output mana yang lebih bisa dipertanggungjawabkan ke customer?
3. Berapa selisih total token antara kedua versi? Apakah CoT worth it untuk use case ini?


Answers:
1. Faktor yang paling terlihat adalah tidak adanya basa basi dan langsung to the point dengan jawabannya.
2. Output dengan CoT, karena lebih jelas
3. Selisih Token adalah 108, melakukan CoT pada konteks ini sangat penting karena tidak banyak informasi tidak jelas.

---

# Section 2 — Self-Check

**Skenario:** Customer test drive **Mitsubishi XForce Ultimate** kemarin tapi belum membuat keputusan pembelian. Buat email follow-up dan minta AI mengkritik draft-nya sendiri.


In [20]:
# --- INITIAL DRAFT ---
initial_prompt = """
Tulis email follow-up dalam bahasa Indonesia untuk customer yang test drive
Mitsubishi XForce Ultimate kemarin tapi belum membuat keputusan pembelian.
Email harus profesional, ramah, dan mendorong tindak lanjut.
"""

initial_draft = generate(initial_prompt, temp=0.7, label="Section 2 — INITIAL DRAFT")



  Section 2 — INITIAL DRAFT
  Params: temp=0.7, top_p=0.9, top_k=40, max_tokens=512
Berikut adalah draf email yang profesional, ramah, dan persuasif yang bisa Anda
gunakan untuk menindaklanjuti calon pembeli Mitsubishi XForce.  ***  **Subjek:
Terima kasih atas kunjungan Anda – Pengalaman berkendara Mitsubishi XForce**
Yth. Bapak/Ibu [Nama Customer],  Terima kasih banyak telah meluangkan waktu
untuk berkunjung ke [Nama Dealer/Showroom] kemarin dan mencoba langsung
pengalaman berkendara dengan **Mitsubishi XForce Ultimate**.  Saya berharap
Bapak/Ibu menikmati kenyamanan, performa, serta fitur-fitur unggulan yang
ditawarkan oleh XForce, terutama [sebutkan fitur spesifik yang sempat dibahas,
misal: sistem audio Yamaha Premium atau mode berkendara yang sesuai dengan
medan].  Saya memahami bahwa memilih kendaraan adalah keputusan penting. Apakah
ada hal spesifik atau keraguan yang ingin Bapak/Ibu diskusikan lebih lanjut
terkait unit tersebut? Saya dengan senang hati akan membantu memberikan

In [21]:
# --- SELF-CHECK + REVISION ---
self_check_prompt = f"""
Berikut adalah draft email follow-up:

---
{initial_draft}
---

Lakukan self-check dengan langkah berikut:
1. Identifikasi 3 kelemahan email tersebut. Pertimbangkan: apakah terlalu pushy?
   Kurang personal? CTA tidak jelas? Tidak menonjolkan keunggulan XForce
   (misal: desain bold, fitur Mitsubishi Connect, ground clearance tinggi)?
2. Tulis ulang versi yang lebih efektif untuk konversi penjualan,
   mempertahankan tone profesional namun lebih hangat dan persuasif.

Format output:

**Kelemahan yang ditemukan:**
1. ...
2. ...
3. ...

**Revised Email:**
[email yang sudah diperbaiki]
"""

generate(self_check_prompt, temp=0.5, label="Section 2 — SELF-CHECK + REVISION")



  Section 2 — SELF-CHECK + REVISION
  Params: temp=0.5, top_p=0.9, top_k=40, max_tokens=512
Berikut adalah hasil *self-check* dan revisi email untuk meningkatkan konversi
penjualan Mitsubishi XForce Anda:  ### **Kelemahan yang ditemukan:**  1.
**Kurang Menonjolkan *Unique Selling Point* (USP):** Email asli terlalu umum.
XForce memiliki keunggulan kompetitif yang kuat seperti *ground clearance*
tertinggi di kelasnya (222mm) dan kenyamanan suspensi yang dirancang khusus
untuk jalanan Indonesia. Hal ini belum dimanfaatkan untuk menciptakan urgensi
atau alasan emosional bagi pembeli. 2.  **CTA (*Call to Action*) Terlalu
Pasif:** Kalimat "Jika ada pertanyaan, silakan hubungi saya" bersifat menunggu.
Untuk konversi yang lebih baik, diperlukan CTA yang lebih spesifik dan
memberikan nilai tambah (misal: penawaran simulasi kredit khusus atau
ketersediaan unit untuk *test drive* di rumah). 3.  **Terlalu Formal dan Kaku:**
Penggunaan bahasa yang terlalu formal terkadang menciptakan jarak antara 

'Berikut adalah hasil *self-check* dan revisi email untuk meningkatkan konversi penjualan Mitsubishi XForce Anda:\n\n### **Kelemahan yang ditemukan:**\n\n1.  **Kurang Menonjolkan *Unique Selling Point* (USP):** Email asli terlalu umum. XForce memiliki keunggulan kompetitif yang kuat seperti *ground clearance* tertinggi di kelasnya (222mm) dan kenyamanan suspensi yang dirancang khusus untuk jalanan Indonesia. Hal ini belum dimanfaatkan untuk menciptakan urgensi atau alasan emosional bagi pembeli.\n2.  **CTA (*Call to Action*) Terlalu Pasif:** Kalimat "Jika ada pertanyaan, silakan hubungi saya" bersifat menunggu. Untuk konversi yang lebih baik, diperlukan CTA yang lebih spesifik dan memberikan nilai tambah (misal: penawaran simulasi kredit khusus atau ketersediaan unit untuk *test drive* di rumah).\n3.  **Terlalu Formal dan Kaku:** Penggunaan bahasa yang terlalu formal terkadang menciptakan jarak antara *sales* dan calon pembeli. Di era sekarang, pendekatan yang lebih *consultative* dan 

In [22]:
# --- SELF-CHECK + REVISION (Duplicated) ---
self_check_prompt = f"""
Berikut adalah draft email follow-up:

---
{initial_draft}
---
Act as a Mitsubishi Sales Manager dengan 10 tahun pengalaman dan kritik ulang

Lakukan self-check dengan langkah berikut:
1. Identifikasi 3 kelemahan email tersebut. Pertimbangkan: apakah terlalu pushy?
   Kurang personal? CTA tidak jelas? Tidak menonjolkan keunggulan XForce
   (misal: desain bold, fitur Mitsubishi Connect, ground clearance tinggi)?
2. Tulis ulang versi yang lebih efektif untuk konversi penjualan,
   mempertahankan tone profesional namun lebih hangat dan persuasif.

Format output:

**Kelemahan yang ditemukan:**
1. ...
2. ...
3. ...

**Revised Email:**
[email yang sudah diperbaiki]
"""

generate(self_check_prompt, temp=0.5, label="Section 2 — SELF-CHECK + REVISION")



  Section 2 — SELF-CHECK + REVISION
  Params: temp=0.5, top_p=0.9, top_k=40, max_tokens=512
Halo, saya telah meninjau draf email Anda. Sebagai Sales Manager dengan
pengalaman 10 tahun, saya melihat draf tersebut sudah cukup baik secara tata
bahasa, namun masih terasa seperti "template standar" yang sering diabaikan oleh
calon pembeli di tengah kesibukan mereka.  Berikut adalah evaluasi dan versi
perbaikannya:  ### Kelemahan yang ditemukan:  1.  **Terlalu Formal dan "Kaku":**
Penggunaan bahasa yang terlalu baku membuat email terasa seperti surat resmi
dari perusahaan, bukan komunikasi personal antara *sales consultant* dan calon
klien. Ini menurunkan tingkat kedekatan (rapport) yang krusial dalam penjualan
otomotif. 2.  **Kurang "Hook" (Nilai Jual Unik):** Email tersebut tidak
menonjolkan *Unique Selling Point* (USP) utama XForce. Calon pembeli butuh
diingatkan kembali mengapa XForce adalah pilihan terbaik dibandingkan kompetitor
(seperti *ground clearance* terbaik di kelasnya atau ken

'Halo, saya telah meninjau draf email Anda. Sebagai Sales Manager dengan pengalaman 10 tahun, saya melihat draf tersebut sudah cukup baik secara tata bahasa, namun masih terasa seperti "template standar" yang sering diabaikan oleh calon pembeli di tengah kesibukan mereka.\n\nBerikut adalah evaluasi dan versi perbaikannya:\n\n### Kelemahan yang ditemukan:\n\n1.  **Terlalu Formal dan "Kaku":** Penggunaan bahasa yang terlalu baku membuat email terasa seperti surat resmi dari perusahaan, bukan komunikasi personal antara *sales consultant* dan calon klien. Ini menurunkan tingkat kedekatan (rapport) yang krusial dalam penjualan otomotif.\n2.  **Kurang "Hook" (Nilai Jual Unik):** Email tersebut tidak menonjolkan *Unique Selling Point* (USP) utama XForce. Calon pembeli butuh diingatkan kembali mengapa XForce adalah pilihan terbaik dibandingkan kompetitor (seperti *ground clearance* terbaik di kelasnya atau kenyamanan kabin yang *best-in-class*).\n3.  **CTA (Call to Action) Terlalu Pasif:** Kal

### 🔍 Reflection — Section 2

1. Kelemahan mana yang paling tajam ditemukan AI?
2. Apakah revised email terasa lebih natural? Atau malah over-corrected?
3. Coba escalate dengan persona: ubah prompt jadi *"Act as a Mitsubishi Sales Manager dengan 10 tahun pengalaman dan kritik ulang"*. Apakah depth analisisnya berbeda?


1. Kurang value proposition, Terlalu pasif, Kurang sentuhan emosional.
2. Sangat over-corrected, especially bagian yang ditambahkan sentuhan emosional. Karena ini kesannya berbisinis, seharusny tidak perlu ditambahkan.
3. Ada, bagian sentuhan emosional diubah menjadi penambahan sense of urgency dimana mendorong sales untuk mengajak recipient membeli mobil.

---

# Section 3 — Retrieval-Aware

**Skenario:** AI hanya boleh menjawab berdasarkan konteks showroom yang diberikan. Test apakah AI menolak hallucinate untuk pertanyaan yang TIDAK ada di konteks.


In [23]:
# --- KONTEKS SHOWROOM ---
showroom_context = """
=== KONTEKS SHOWROOM (Maret 2026) ===
Mitsubishi Showroom Jakarta:
- Xpander Ultimate 2024 A/T: 4 unit, harga Rp 295jt
- Xpander Cross Premium 2024: 3 unit, harga Rp 335jt
- XForce Ultimate 2024: 5 unit, harga Rp 405jt
- Pajero Sport Dakar 4x2 2024: 2 unit, harga Rp 720jt

Promo bulan ini:
- DP 10% untuk Xpander
- Gratis service 1 tahun untuk XForce
- Bonus aksesoris Rp 15jt untuk Pajero Sport

Jam operasional: Senin-Sabtu 09:00-18:00.
=== END KONTEKS ===
"""

retrieval_aware_prompt = f"""
{showroom_context}

INSTRUKSI:
Jawab pertanyaan customer HANYA berdasarkan konteks di atas.
Jika informasi tidak ada di konteks, jawab persis:
"Informasi tidak tersedia dalam sumber. Silakan hubungi sales advisor untuk detail lebih lanjut."
Sertakan kutipan kalimat pendukung dari konteks untuk setiap jawaban.

PERTANYAAN CUSTOMER:
1. Berapa unit XForce Ultimate yang tersedia?
2. Promo apa yang berlaku untuk Xpander bulan ini?
3. Apakah showroom buka hari Minggu?
4. Apakah ada promo untuk Mitsubishi Triton bulan ini?

Format jawaban untuk SETIAP pertanyaan:
- Pertanyaan: [pertanyaan]
- Jawaban: [jawaban]
- Sumber: "[kutipan langsung dari konteks ATAU 'Tidak tersedia di konteks']"
"""

generate(retrieval_aware_prompt, temp=0.1, label="Section 3 — RETRIEVAL-AWARE")



  Section 3 — RETRIEVAL-AWARE
  Params: temp=0.1, top_p=0.9, top_k=40, max_tokens=512
Berikut adalah jawaban berdasarkan konteks yang diberikan:  - Pertanyaan: Berapa
unit XForce Ultimate yang tersedia? - Jawaban: Saat ini tersedia 5 unit XForce
Ultimate 2024. - Sumber: "XForce Ultimate 2024: 5 unit, harga Rp 405jt"  -
Pertanyaan: Promo apa yang berlaku untuk Xpander bulan ini? - Jawaban: Promo
yang berlaku untuk Xpander bulan ini adalah DP 10%. - Sumber: "DP 10% untuk
Xpander"  - Pertanyaan: Apakah showroom buka hari Minggu? - Jawaban: Tidak,
showroom hanya beroperasi pada hari Senin sampai Sabtu. - Sumber: "Jam
operasional: Senin-Sabtu 09:00-18:00."  - Pertanyaan: Apakah ada promo untuk
Mitsubishi Triton bulan ini? - Jawaban: Informasi tidak tersedia dalam sumber.
Silakan hubungi sales advisor untuk detail lebih lanjut. - Sumber: "Tidak
tersedia di konteks"

📊 Tokens — input: 365, output: 221, total: 586


'Berikut adalah jawaban berdasarkan konteks yang diberikan:\n\n- Pertanyaan: Berapa unit XForce Ultimate yang tersedia?\n- Jawaban: Saat ini tersedia 5 unit XForce Ultimate 2024.\n- Sumber: "XForce Ultimate 2024: 5 unit, harga Rp 405jt"\n\n- Pertanyaan: Promo apa yang berlaku untuk Xpander bulan ini?\n- Jawaban: Promo yang berlaku untuk Xpander bulan ini adalah DP 10%.\n- Sumber: "DP 10% untuk Xpander"\n\n- Pertanyaan: Apakah showroom buka hari Minggu?\n- Jawaban: Tidak, showroom hanya beroperasi pada hari Senin sampai Sabtu.\n- Sumber: "Jam operasional: Senin-Sabtu 09:00-18:00."\n\n- Pertanyaan: Apakah ada promo untuk Mitsubishi Triton bulan ini?\n- Jawaban: Informasi tidak tersedia dalam sumber. Silakan hubungi sales advisor untuk detail lebih lanjut.\n- Sumber: "Tidak tersedia di konteks"'

In [24]:
# --- KONTEKS SHOWROOM (Diubah Temperature nya ke 1) ---
showroom_context = """
=== KONTEKS SHOWROOM (Maret 2026) ===
Mitsubishi Showroom Jakarta:
- Xpander Ultimate 2024 A/T: 4 unit, harga Rp 295jt
- Xpander Cross Premium 2024: 3 unit, harga Rp 335jt
- XForce Ultimate 2024: 5 unit, harga Rp 405jt
- Pajero Sport Dakar 4x2 2024: 2 unit, harga Rp 720jt

Promo bulan ini:
- DP 10% untuk Xpander
- Gratis service 1 tahun untuk XForce
- Bonus aksesoris Rp 15jt untuk Pajero Sport

Jam operasional: Senin-Sabtu 09:00-18:00.
=== END KONTEKS ===
"""

retrieval_aware_prompt = f"""
{showroom_context}

INSTRUKSI:
Jawab pertanyaan customer HANYA berdasarkan konteks di atas.
Jika informasi tidak ada di konteks, jawab persis:
"Informasi tidak tersedia dalam sumber. Silakan hubungi sales advisor untuk detail lebih lanjut."
Sertakan kutipan kalimat pendukung dari konteks untuk setiap jawaban.

PERTANYAAN CUSTOMER:
1. Berapa unit XForce Ultimate yang tersedia?
2. Promo apa yang berlaku untuk Xpander bulan ini?
3. Apakah showroom buka hari Minggu?
4. Apakah ada promo untuk Mitsubishi Triton bulan ini?

Format jawaban untuk SETIAP pertanyaan:
- Pertanyaan: [pertanyaan]
- Jawaban: [jawaban]
- Sumber: "[kutipan langsung dari konteks ATAU 'Tidak tersedia di konteks']"
"""

generate(retrieval_aware_prompt, temp=1.0, label="Section 3 — RETRIEVAL-AWARE")



  Section 3 — RETRIEVAL-AWARE
  Params: temp=1.0, top_p=0.9, top_k=40, max_tokens=512
Berikut adalah jawaban berdasarkan konteks yang diberikan:  - Pertanyaan: Berapa
unit XForce Ultimate yang tersedia? - Jawaban: Tersedia 5 unit XForce Ultimate
2024. - Sumber: "XForce Ultimate 2024: 5 unit, harga Rp 405jt"  - Pertanyaan:
Promo apa yang berlaku untuk Xpander bulan ini? - Jawaban: Promo untuk Xpander
bulan ini adalah DP 10%. - Sumber: "DP 10% untuk Xpander"  - Pertanyaan: Apakah
showroom buka hari Minggu? - Jawaban: Tidak, showroom hanya beroperasi pada hari
Senin hingga Sabtu. - Sumber: "Jam operasional: Senin-Sabtu 09:00-18:00."  -
Pertanyaan: Apakah ada promo untuk Mitsubishi Triton bulan ini? - Jawaban:
Informasi tidak tersedia dalam sumber. Silakan hubungi sales advisor untuk
detail lebih lanjut. - Sumber: "Tidak tersedia di konteks"

📊 Tokens — input: 365, output: 218, total: 583


'Berikut adalah jawaban berdasarkan konteks yang diberikan:\n\n- Pertanyaan: Berapa unit XForce Ultimate yang tersedia?\n- Jawaban: Tersedia 5 unit XForce Ultimate 2024.\n- Sumber: "XForce Ultimate 2024: 5 unit, harga Rp 405jt"\n\n- Pertanyaan: Promo apa yang berlaku untuk Xpander bulan ini?\n- Jawaban: Promo untuk Xpander bulan ini adalah DP 10%.\n- Sumber: "DP 10% untuk Xpander"\n\n- Pertanyaan: Apakah showroom buka hari Minggu?\n- Jawaban: Tidak, showroom hanya beroperasi pada hari Senin hingga Sabtu.\n- Sumber: "Jam operasional: Senin-Sabtu 09:00-18:00."\n\n- Pertanyaan: Apakah ada promo untuk Mitsubishi Triton bulan ini?\n- Jawaban: Informasi tidak tersedia dalam sumber. Silakan hubungi sales advisor untuk detail lebih lanjut.\n- Sumber: "Tidak tersedia di konteks"'

### 🔍 Reflection — Section 3

1. Apakah AI menolak halusinasi promo Triton (yang tidak ada di konteks)?
2. Apakah kutipan sumber yang diberikan benar-benar dari konteks atau ada yang dikarang?
3. Apa yang terjadi jika `temp` dinaikkan ke 1.0? Apakah AI tetap patuh?


1. Iya menolak halusinasi promo Triton
2. Benar semua
3. Tidak terjadi perubahan yang signifikan dan AI tetap patuh pada konteks yang telah diberikan

---

# Section 4 — Final Assignment: Combined Techniques

## 🎯 Objective
Build an optimized prompt for ONE Mitsubishi business use case. Your final prompt **MUST** combine:

| Layer | Pick At Least |
|---|---|
| Reasoning technique | **1 of:** CoT / Self-Check / Retrieval-Aware |
| Engine parameters | **All 3:** Temperature, Top-P, Top-K |
| Cost optimization | **1 of:** Token Budgeting (`max_tokens`) / Prompt Compression |

## 🚗 Choose ONE Use Case
1. **Sales pitch generator** — pitch text untuk customer pada Mitsubishi model tertentu
2. **Lead nurturing email** — follow-up email untuk test drive prospect
3. **Marketplace listing** — deskripsi used Mitsubishi untuk listing online
4. **Customer service response** — balasan untuk komplain delayed service appointment

## 📦 Dummy Data (Mitsubishi)


In [25]:
DUMMY_DATA = """
=== INVENTORY (Mitsubishi Showroom) ===
- Xpander Ultimate 2024 A/T: 4 unit, KM 8.000, Rp 295jt
- Xpander Cross Premium 2024: 3 unit, KM 5.000, Rp 335jt
- XForce Ultimate 2024: 5 unit, KM 3.500, Rp 405jt
- Pajero Sport Dakar 4x2 2024: 2 unit, KM 6.000, Rp 720jt
- Triton Athlete 4x4 2023: 2 unit, KM 18.000, Rp 525jt

=== CUSTOMER PROFILE ===
- Nama: Budi Santoso
- Usia: 35 tahun
- Keluarga: 4 orang (istri + 2 anak)
- Budget: Rp 280-340jt
- Kebutuhan: MPV/crossover hemat BBM untuk harian + sesekali keluar kota

=== SERVICE CENTER ===
- Avg waktu servis: 2-3 jam
- Ganti oli: Rp 450k
- Tune-up: Rp 950k
- Paket servis berkala: kelipatan KM 10.000
"""

print(DUMMY_DATA)



=== INVENTORY (Mitsubishi Showroom) ===
- Xpander Ultimate 2024 A/T: 4 unit, KM 8.000, Rp 295jt
- Xpander Cross Premium 2024: 3 unit, KM 5.000, Rp 335jt
- XForce Ultimate 2024: 5 unit, KM 3.500, Rp 405jt
- Pajero Sport Dakar 4x2 2024: 2 unit, KM 6.000, Rp 720jt
- Triton Athlete 4x4 2023: 2 unit, KM 18.000, Rp 525jt

=== CUSTOMER PROFILE ===
- Nama: Budi Santoso
- Usia: 35 tahun
- Keluarga: 4 orang (istri + 2 anak)
- Budget: Rp 280-340jt
- Kebutuhan: MPV/crossover hemat BBM untuk harian + sesekali keluar kota

=== SERVICE CENTER ===
- Avg waktu servis: 2-3 jam
- Ganti oli: Rp 450k
- Tune-up: Rp 950k
- Paket servis berkala: kelipatan KM 10.000



## ✏️ Your Final Prompt

Edit the cell below — pilih use case, tentukan teknik reasoning, lalu tulis prompt Anda.


In [26]:
# ============================================================
# TODO: Lengkapi prompt final Anda di bawah
# Pastikan kombinasi berikut ada:
#   - 1 reasoning technique (CoT / Self-Check / Retrieval-Aware)
#   - All 3 engine params (temp, top_p, top_k) — di-set di Section 5
#   - 1 cost optimization (max_tokens = token budgeting,
#     atau compress prompt agar lebih ringkas)
# ============================================================

YOUR_USE_CASE = "Customer service response"   # ← ganti sesuai pilihan Anda
YOUR_REASONING_TECHNIQUE = "CoT"          # ← CoT / Self-Check / Retrieval-Aware
YOUR_COST_STRATEGY = "Prompt Compression"    # ← Token Budgeting / Prompt Compression

YOUR_FINAL_PROMPT = f"""
The dummy data for context is available below
{DUMMY_DATA}
--------
Act as a Customer Service employee in a professional tone, clear explanation, and give efficient solution.

Reasoning:
- Identify customers main problem
- Offer relevant solutions
- Create a short and polite answer
Do not show the thinking process, only the final answer

Rules:
- Answer with the users input language
- Generate with only 120 maximum words
- Do not generate answer that are out of context
- If you need more data, proceed to the validation process
- Prioritize empathy, proffesionalism, and efficient solution
- If you can not answer the question, translate 'Sorry I could not answer that, proceed to contact customer support' in the users language

Output format:
<Final Answer>
"""

print("===== USE CASE =====")
print(YOUR_USE_CASE)
print("\n===== REASONING =====")
print(YOUR_REASONING_TECHNIQUE)
print("\n===== COST STRATEGY =====")
print(YOUR_COST_STRATEGY)
print("\n===== FINAL PROMPT =====")
print(YOUR_FINAL_PROMPT)


===== USE CASE =====
Customer service response

===== REASONING =====
CoT

===== COST STRATEGY =====
Prompt Compression

===== FINAL PROMPT =====

The dummy data for context is available below

=== INVENTORY (Mitsubishi Showroom) ===
- Xpander Ultimate 2024 A/T: 4 unit, KM 8.000, Rp 295jt
- Xpander Cross Premium 2024: 3 unit, KM 5.000, Rp 335jt
- XForce Ultimate 2024: 5 unit, KM 3.500, Rp 405jt
- Pajero Sport Dakar 4x2 2024: 2 unit, KM 6.000, Rp 720jt
- Triton Athlete 4x4 2023: 2 unit, KM 18.000, Rp 525jt

=== CUSTOMER PROFILE ===
- Nama: Budi Santoso
- Usia: 35 tahun
- Keluarga: 4 orang (istri + 2 anak)
- Budget: Rp 280-340jt
- Kebutuhan: MPV/crossover hemat BBM untuk harian + sesekali keluar kota

=== SERVICE CENTER ===
- Avg waktu servis: 2-3 jam
- Ganti oli: Rp 450k
- Tune-up: Rp 950k
- Paket servis berkala: kelipatan KM 10.000

--------
Act as a Customer Service employee in a professional tone, clear explanation, and give efficient solution.

Reasoning:
- Identify customers main p

---

# Section 5 — Run 3 Configurations

Keep the prompt **constant**. Vary only the engine parameters. The reasoning technique stays the same — you're isolating the parameter effect.


In [27]:
EXPERIMENTS = [
    {"label": "Exp 1 — Safe",     "temp": 0.2, "top_p": 0.5, "top_k": 20,  "max_tokens": 200},
    {"label": "Exp 2 — Balanced", "temp": 0.6, "top_p": 0.8, "top_k": 50,  "max_tokens": 400},
    {"label": "Exp 3 — Creative", "temp": 1.0, "top_p": 1.0, "top_k": 100, "max_tokens": 600},
]

results = {}
for exp in EXPERIMENTS:
    text = generate(
        YOUR_FINAL_PROMPT,
        temp=exp["temp"],
        top_p=exp["top_p"],
        top_k=exp["top_k"],
        max_tokens=exp["max_tokens"],
        label=exp["label"],
    )
    results[exp["label"]] = text

print("\n\n✅ All 3 experiments completed. Hasil tersimpan di dict `results`.")



  Exp 1 — Safe
  Params: temp=0.2, top_p=0.5, top_k=20, max_tokens=200
Halo Bapak Budi Santoso, terima kasih telah menghubungi kami.  Mempertimbangkan
kebutuhan Bapak akan kendaraan keluarga yang hemat BBM untuk harian dan
perjalanan luar kota dengan budget Rp 280-340 juta, saya merekomendasikan
**Xpander Cross Premium 2024**.  Unit ini sangat cocok untuk keluarga dengan 4
anggota, memiliki kenyamanan MPV, serta ketangguhan crossover untuk perjalanan
luar kota. Saat ini kami memiliki 3 unit dengan KM rendah (5.000) seharga Rp 335
juta, yang sangat sesuai dengan anggaran Bapak.  Apakah Bapak bersedia
meluangkan waktu untuk melakukan *test drive* minggu ini? Saya siap membantu
mengatur jadwal Bapak. Jika ada pertanyaan lebih lanjut mengenai spesifikasi
atau skema kredit, silakan sampaikan kepada saya.

📊 Tokens — input: 468, output: 168, total: 636

  Exp 2 — Balanced
  Params: temp=0.6, top_p=0.8, top_k=50, max_tokens=400
Halo Pak Budi Santoso, terima kasih telah menghubungi kami.  Mel

In [28]:
# ============================================================
# TODO: Lengkapi prompt final Anda di bawah
# Pastikan kombinasi berikut ada:
#   - 1 reasoning technique (CoT / Self-Check / Retrieval-Aware)
#   - All 3 engine params (temp, top_p, top_k) — di-set di Section 5
#   - 1 cost optimization (max_tokens = token budgeting,
#     atau compress prompt agar lebih ringkas)
# ============================================================

YOUR_USE_CASE = "Customer service response"   # ← ganti sesuai pilihan Anda
YOUR_REASONING_TECHNIQUE = "Self-Check"          # ← CoT / Self-Check / Retrieval-Aware
YOUR_COST_STRATEGY = "Prompt Compression"    # ← Token Budgeting / Prompt Compression

YOUR_FINAL_PROMPT = f"""
The dummy data for context is available below
{DUMMY_DATA}
--------
Act as a Customer Service employee in a professional tone, clear explanation, and give efficient solution.

Reasoning:
- Identify customers main problem
- Offer relevant solutions
- Create a short and polite answer
Do not show the thinking process, only the final answer

Rules:
- Answer with the users input language
- Generate with only 120 maximum words
- Do not generate answer that are out of context
- If you need more data, proceed to the validation process
- Prioritize empathy, proffesionalism, and efficient solution
- If you can not answer the question, translate 'Sorry I could not answer that, proceed to contact customer support' in the users language

Output format:
<Final Answer>
"""

print("===== USE CASE =====")
print(YOUR_USE_CASE)
print("\n===== REASONING =====")
print(YOUR_REASONING_TECHNIQUE)
print("\n===== COST STRATEGY =====")
print(YOUR_COST_STRATEGY)
print("\n===== FINAL PROMPT =====")
print(YOUR_FINAL_PROMPT)


===== USE CASE =====
Customer service response

===== REASONING =====
Self-Check

===== COST STRATEGY =====
Prompt Compression

===== FINAL PROMPT =====

The dummy data for context is available below

=== INVENTORY (Mitsubishi Showroom) ===
- Xpander Ultimate 2024 A/T: 4 unit, KM 8.000, Rp 295jt
- Xpander Cross Premium 2024: 3 unit, KM 5.000, Rp 335jt
- XForce Ultimate 2024: 5 unit, KM 3.500, Rp 405jt
- Pajero Sport Dakar 4x2 2024: 2 unit, KM 6.000, Rp 720jt
- Triton Athlete 4x4 2023: 2 unit, KM 18.000, Rp 525jt

=== CUSTOMER PROFILE ===
- Nama: Budi Santoso
- Usia: 35 tahun
- Keluarga: 4 orang (istri + 2 anak)
- Budget: Rp 280-340jt
- Kebutuhan: MPV/crossover hemat BBM untuk harian + sesekali keluar kota

=== SERVICE CENTER ===
- Avg waktu servis: 2-3 jam
- Ganti oli: Rp 450k
- Tune-up: Rp 950k
- Paket servis berkala: kelipatan KM 10.000

--------
Act as a Customer Service employee in a professional tone, clear explanation, and give efficient solution.

Reasoning:
- Identify customers

In [29]:
EXPERIMENTS = [
    {"label": "Exp 1 — Safe",     "temp": 0.2, "top_p": 0.5, "top_k": 20,  "max_tokens": 200},
    {"label": "Exp 2 — Balanced", "temp": 0.6, "top_p": 0.8, "top_k": 50,  "max_tokens": 400},
    {"label": "Exp 3 — Creative", "temp": 1.0, "top_p": 1.0, "top_k": 100, "max_tokens": 600},
]

results = {}
for exp in EXPERIMENTS:
    text = generate(
        YOUR_FINAL_PROMPT,
        temp=exp["temp"],
        top_p=exp["top_p"],
        top_k=exp["top_k"],
        max_tokens=exp["max_tokens"],
        label=exp["label"],
    )
    results[exp["label"]] = text

print("\n\n✅ All 3 experiments completed. Hasil tersimpan di dict `results`.")



  Exp 1 — Safe
  Params: temp=0.2, top_p=0.5, top_k=20, max_tokens=200
Halo Bapak Budi Santoso, terima kasih telah menghubungi kami.  Mempertimbangkan
kebutuhan Bapak akan kendaraan keluarga yang hemat BBM untuk harian dan
perjalanan luar kota dengan budget Rp 280-340 juta, saya merekomendasikan
**Xpander Cross Premium 2024**.  Unit ini sangat cocok untuk kapasitas 4 orang,
memiliki kenyamanan MPV, serta ketangguhan crossover untuk medan luar kota. Saat
ini kami memiliki 3 unit dengan KM rendah (5.000) seharga Rp 335 juta, yang
masih masuk dalam rentang anggaran Bapak.  Apakah Bapak bersedia meluangkan
waktu untuk melakukan *test drive* minggu ini? Saya siap membantu
menjadwalkannya untuk Bapak.

📊 Tokens — input: 468, output: 152, total: 620

  Exp 2 — Balanced
  Params: temp=0.6, top_p=0.8, top_k=50, max_tokens=400
Halo Bapak Budi Santoso, terima kasih telah menghubungi kami.  Melihat kebutuhan
Bapak untuk mobil keluarga (4 orang) yang hemat BBM untuk harian maupun luar
kota dengan 

---

# Section 6 — Analysis Table

Score 1–5 dengan catatan singkat untuk setiap sel.
Edit sel markdown di bawah dan isi tabelnya.

| Criteria | Exp 1 (Safe) | Exp 2 (Balanced) | Exp 3 (Creative) |
|---|---|---|---|
| Clarity | 4 | 3 | 5 |
| Creativity | 4 | 1 | 4 |
| Consistency *(jalankan 2x, bandingkan)* | 4 | 4 | 3 |
| Reasoning Quality *(efektivitas CoT/Self-Check)* | 4 | 3 | 3 |
| Groundedness *(kepatuhan ke konteks, jika Retrieval-Aware)* | - | - | - |
| Token Efficiency *(total tokens dari output)* | 3 | 2 | 4 |
| **Best Output** *(pilih satu)* | 1 |  |  |


---

# Section 7 — Reflection (200–300 kata)

_(Tulis refleksi Anda di sel markdown ini)_

Pertanyaan panduan:
1. Config mana yang menang untuk use case Anda? Mengapa?
2. Teknik reasoning mana yang paling impactful — CoT, Self-Check, atau Retrieval-Aware?
3. Apa yang mengejutkan dari hasil eksperimen?
4. Bagaimana cost optimization (max_tokens / compression) berdampak pada kualitas output?
5. Untuk production deployment di Mitsubishi, config mana yang akan Anda rekomendasikan? Apa risikonya?

**[Tulis refleksi 200-300 kata di sini]**


&nbsp;

Config yang menurut saya paling bagus adalah yang Safe dalam configurasi Chain of Thought. Hal ini karena banyak hal yang lebih sesuai dengan prompt yang sudah ditentukan sehingga menghasilkan jawaban yang lebih impactful dan sesuai untuk pertanyaan user. Yang mengejutkan adalah Creative lebih bagus daripada yang Balanced, bagian balanced lebih cenderung memilih kata-kata yang kurang cocok dengan konteks pertanyaan. Sedangkan bagian Creative masih dapat memberikan jawaban yang sesuai dengan konteks walaupun pemilihan kata-kata tidak secocok config Safe. Cost yang diberikan dari masing-masing config sudah cukup baik, tetapi terkadang yang balance itu lebih murah daripada yang safe. Namun, hasil yang diberikan walaupun sedikit lebih mahal sangat signifikan perbedaannya. Maka dari itu, untuk production, config yang akan saya rekomendasikan adalah menggunakan config Safe dengan Chain of Thought

---

# ✅ Submission Checklist

Sebelum submit ke Google Classroom, pastikan:

- [x] Setup berhasil — API key dari Colab Secrets, **TIDAK** hardcoded
- [x] **Section 1 (CoT)** — kedua versi (without/with) sudah dijalankan
- [x] **Section 2 (Self-Check)** — initial draft + revised version sudah dijalankan
- [x] **Section 3 (Retrieval-Aware)** — keempat pertanyaan dijawab dengan citation
- [x] **Section 4** — `YOUR_USE_CASE`, `YOUR_REASONING_TECHNIQUE`, `YOUR_COST_STRATEGY`, dan `YOUR_FINAL_PROMPT` sudah diisi
- [x] **Section 5** — 3 eksperimen sudah dijalankan
- [x] **Section 6** — analysis table terisi lengkap (tidak ada sel kosong)
- [x] **Section 7** — refleksi 200–300 kata sudah ditulis
- [x] File rename: `NamaLengkap_Sesi25_Assignment.ipynb`
- [x] Upload ke Google Classroom sebelum deadline

**Selamat mengerjakan! 🚗💨**
